# <center>**Enhancing LEM-X Imaging with the IROS Reconstruction Pipeline**<center>

## <center>**Sky Reconstruction Efficiency**<center>

In [1]:
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.io import simulation_files
from bloodmoon.types import CoordEquatorial
import darksun as ds
from darksun.data import Log, DataLoader, CatalogueLoader

from IROSrec.handle import config_dirpaths
import imgmaker as mgm
from imgmaker.fns import CameraUnitMap

In [2]:
MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"

# SKYFIELD: str = "GalacticCentre"
SKYFIELD: str = "IROSDummy"
DATA_FITS: str = "baseline_2-50keV_1ks"

RUN_ID: str = 'IROSbenchmrk_upx5upy1_detected_2-5keV_noS17'

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "detected"

E_min: float = 2.0  # [keV]
E_max: float = 5.0  # [keV]
coords2exclude: list[CoordEquatorial] | None = [CoordEquatorial(239.824508666992, -54.0559692382813)]

UP_X, UP_Y = 5, 1

In [3]:
MASK_PATH, SIMUL_DATA_PATH, SAVE_PATH = config_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
    runID=RUN_ID,
)
OUT_RESULTS_PATH = mgm.config_savedata_to()

wfm: CodedMaskCamera = codedmask(MASK_PATH, UP_X, UP_Y)

filepaths: dict[str, dict[str, Path]] = simulation_files(SIMUL_DATA_PATH)
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

logA, logB = ds.load_database(f"{SAVE_PATH}/IROS_sources_db.fits")

# Loading data...
# Loading completed!


CATALOG ='IROS_benchmark_catalogue_1ks.fits' / Catalog file                      [astropy.io.fits.card]


### <center>**Benchmark Tables**<center>

In [4]:
import re

def adjust_Tabfrmt(txt: str) -> str:
    # insert \hline instead of rules (journal guidelines)
    for rule in ('toprule', 'midrule', 'bottomrule'):
        txt = txt.replace(rule, 'hline')
    # shift caption and label at the end (journal guidelines)
    pattern = r"(\\begin\{table\}.*?)(\\caption\{.*?\})\s*(\\label\{.*?\})\s*(\\begin\{tabular\}.*?\\end\{tabular\})"
    replacement = r"\1\4\n\2\n\3"
    txt = re.sub(pattern, replacement, txt, flags=re.DOTALL)
    # convert to onecolumn
    txt = txt.replace('table', 'table*')
    return txt

def sort_by(df: pd.DataFrame, key: str, **kwargs: Any) -> pd.DataFrame:
    """Sort DataFrame wrt input column key."""
    return df.sort_values(by=[key], ascending=False, ignore_index=True, **kwargs)

In [5]:
from numpy.typing import NDArray

def gather_cam_data(
    log: Log,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
    varmap: NDArray,
) -> pd.DataFrame:
    """
    Gathers single camera data from IROS reconstruction database.
    """
    ids = np.array([src.upper() for src in log.log['ID']])
    theta_res_x, theta_res_y = mgm.get_angularcoords_residues(
        log, catalogue, sdl, camera,
    )
    cts = np.array(log.log['fluence'])
    true_cts = mgm.extract_catalogue_fluences(log, catalogue, sdl, camera)
    n, m = varmap.shape
    boxsize = (80, 200)
    srows, scols = (
        slice((n - 1) // 2 - camera.upscale_f.y * boxsize[0], (n - 1) // 2 + camera.upscale_f.y * boxsize[0] + 1),
        slice((m - 1) // 2 - camera.upscale_f.x * boxsize[1], (m - 1) // 2 + camera.upscale_f.x * boxsize[1] + 1),
    )
    rmse = np.sqrt(np.mean(varmap[srows, scols]))
    dmap = {
        log.name: {
            'Source': ids,
            'DthetaX': theta_res_x,
            'DthetaY': theta_res_y,
            'IROS_cts': cts,
            'True_cts': true_cts,
            'Dcts': (cts - true_cts) / np.sqrt(true_cts),
            'Dcts_var': (cts - true_cts) / rmse,
            'SNR': np.array(log.log['snr']),
            # 'thetaX [deg]': np.array(log.log['angle_x']),
            # 'thetaY [deg]': np.array(log.log['angle_y']),
        }
    }
    return pd.DataFrame(dmap)

def get_joint_tab(
    data_camA: pd.DataFrame,
    data_camB: pd.DataFrame,
    unitmap: CameraUnitMap,
) -> pd.DataFrame:
    """Generates a Dataframe with output data from both cameras."""
    compose: Callable = lambda a, b: np.sqrt(a ** 2 + b ** 2)
    dmap = {
        'Source': np.array(data_camA.CAM1A['Source'])[unitmap.idx_a],

        'DthetaX_A': np.array(data_camA.CAM1A['DthetaX'])[unitmap.idx_a],
        'DthetaY_A': np.array(data_camA.CAM1A['DthetaY'])[unitmap.idx_a],
        'TrueCts_A': np.array(data_camA.CAM1A['True_cts'])[unitmap.idx_a],
        'ReconstrCts_A': np.array(data_camA.CAM1A['IROS_cts'])[unitmap.idx_a],
        'Dcts_A': np.array(data_camA.CAM1A['Dcts'])[unitmap.idx_a],
        'Dcts_A_var': np.array(data_camA.CAM1A['Dcts_var'])[unitmap.idx_a],

        'DthetaX_B': np.array(data_camB.CAM1B['DthetaX'])[unitmap.idx_b],
        'DthetaY_B': np.array(data_camB.CAM1B['DthetaY'])[unitmap.idx_b],
        'TrueCts_B': np.array(data_camB.CAM1B['True_cts'])[unitmap.idx_b],
        'ReconstrCts_B': np.array(data_camB.CAM1B['IROS_cts'])[unitmap.idx_b],
        'Dcts_B': np.array(data_camB.CAM1B['Dcts'])[unitmap.idx_b],
        'Dcts_B_var': np.array(data_camB.CAM1B['Dcts_var'])[unitmap.idx_b],

        'SNR': compose(
            np.array(data_camA.CAM1A['SNR'])[unitmap.idx_a],
            np.array(data_camB.CAM1B['SNR'])[unitmap.idx_b],
        ),
    }
    return pd.DataFrame(dmap)

In [6]:
from bloodmoon.mask import count, variance

def get_varmap(camera: CodedMaskCamera, sdl: DataLoader) -> NDArray:
    detector = count(camera, sdl.DLdata)[0]
    varmap = variance(camera, detector)
    return varmap


varmapA, varmapB = map(lambda sdl: get_varmap(wfm, sdl), (sdlA, sdlB))

In [7]:
ds.pixels_angular_resolution(wfm)
cu_map = mgm.get_srcmap_for_unit(logA.log['ID'], logB.log['ID'])

# Table - CAMERA A
data_camA = gather_cam_data(logA, catA, sdlA, wfm, varmapA)

# Table - CAMERA B
data_camB = gather_cam_data(logB, catB, sdlB, wfm, varmapB)


Pixel angular resolution at upscaling (x, y): (5, 1)
  - fine direction: 0.8465 arcmin
  - coarse direction: 8.4653 arcmin



Analysing S29:   0%|          | 0/25 [00:00<?, ?it/s]WARNING: The following header keyword is invalid or follows an unrecognized non-standard convention:
CATALOG ='IROS_benchmark_catalogue_1ks.fits' / Catalog file                      [astropy.io.fits.card]
Analysing S4: 100%|██████████| 25/25 [00:01<00:00, 19.14it/s] 


In [8]:
unit_data = get_joint_tab(data_camA, data_camB, cu_map)

KWS = {
    'label': 'Table1',
    'caption': 'Testing $`to\\_latex`$ fn.',
    'float_format': "%.1f",
    'column_format': 'l' + 'c' * (len(unit_data.columns) - 2) + 'r',
}
tab = mgm.df2TeXtab(
    df=sort_by(unit_data, 'SNR'),
    adjust_tabfrmt=adjust_Tabfrmt,
    save_to=f'{OUT_RESULTS_PATH}/../texTable_Unit_results_{DATASET}_{E_min}-{E_max}_noSCOX1.tex',
    overwrite=True,
    **KWS,
)

In [9]:
unit_data.sort_values('SNR', ascending=False, ignore_index=True)

,Source,DthetaX_A,DthetaY_A,TrueCts_A,ReconstrCts_A,Dcts_A,Dcts_A_var,DthetaX_B,DthetaY_B,TrueCts_B,ReconstrCts_B,Dcts_B,Dcts_B_var,SNR
0,S29,0.009170,2.624296,116328.0,117623.758730,3.799111,1.315603,-0.021637,-1.728712,131252.0,133387.353796,5.894092,2.121656,213.780063
1,S25,0.023762,-2.976841,110671.0,111325.374860,1.967024,0.664396,0.042259,0.356661,101354.0,97841.356154,-11.033509,-3.490111,152.659023
2,S19,0.031009,-1.586369,72442.0,75303.083754,10.630051,2.904900,-0.014390,-1.409821,78071.0,81677.242082,12.906545,3.583109,107.850408
3,S11,-0.011913,-0.665253,52094.0,47375.923378,-20.671459,-4.790331,-0.041825,-3.062815,58923.0,60836.586432,7.883257,1.901311,84.551629
4,S26,-0.123878,0.266128,56604.0,54434.854745,-9.117275,-2.202365,0.046418,-0.001067,56898.0,54440.497533,-10.302567,-2.441738,83.772536
5,S24,-0.124283,-1.177965,52192.0,48632.411869,-15.581088,-3.614101,0.000124,-4.391334,58493.0,59193.559578,2.896631,0.696066,82.300245
6,S12,0.009966,-2.841421,43248.0,42146.759323,-5.295406,-1.118106,-0.052964,3.585440,42384.0,43614.741191,5.978136,1.222846,72.701287
7,S13,-0.138024,2.826699,47845.0,46658.892516,-5.422577,-1.204272,0.048653,-1.074799,49000.0,49053.475041,0.241576,0.053132,70.853762
8,S2,-0.149246,-0.491567,33193.0,33965.982035,4.242737,0.784820,-0.017250,-0.555928,42683.0,44672.290984,9.628771,1.976530,70.584875
9,S27,-0.078615,-1.582678,45892.0,47303.860972,6.590577,1.433483,-0.156735,2.924045,43701.0,43719.642139,0.089176,0.018523,64.203630
